# Employee Wellness Management Analytics

## Milestone 1 - User Authentication Module

### Objective

Develop a secure authentication system using Streamlit and Neon PostgreSQL.

### Technologies Used

- Python
- Streamlit
- Neon PostgreSQL
- bcrypt
- JWT
- Google SMTP

In [ ]:
def get_connection():
    try:
        conn = psycopg2.connect(
            host=os.getenv("DB_HOST"),
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            port=os.getenv("DB_PORT")
        )
        return conn

    except Exception as e:
        print("Database Connection Error:", e)
        return None



## Database Connection

This section establishes a secure connection to the Neon PostgreSQL database using the psycopg2 library.

In [ ]:
def create_users_table():
    conn = get_connection()

    if conn:
        cur = conn.cursor()

        cur.execute("""
        CREATE TABLE IF NOT EXISTS users(
            id SERIAL PRIMARY KEY,
            name VARCHAR(100) NOT NULL,
            email VARCHAR(100) UNIQUE NOT NULL,
            password TEXT NOT NULL
        );
        """)

        conn.commit()
        cur.close()
        conn.close()

## User Registration

The application validates user input, hashes passwords using bcrypt, and stores user details securely in PostgreSQL.

In [ ]:
elif page == "Sign Up":

    st.header("👤 Sign Up")

    name = st.text_input("Name")
    email = st.text_input("Email")
    password = st.text_input("Password", type="password")
    confirm = st.text_input("Confirm Password", type="password")

    if st.button("Register"):

        if name == "" or email == "" or password == "":
            st.error("All fields are required")

        elif password != confirm:
            st.error("Passwords do not match")

        elif get_user(email):
            st.error("Email already registered")

        else:
            hashed = bcrypt.hashpw(
                password.encode(),
                bcrypt.gensalt()
            ).decode()

            add_user(name, email, hashed)

            st.success("Registration Successful!")


## User Login

The application authenticates users using their registered email and hashed password. Upon successful authentication, a JWT token is generated.

In [ ]:
elif page == "Login":

    st.header("🔑 Login")

    email = st.text_input("Email")
    password = st.text_input("Password", type="password")

    if st.button("Login"):

        user = get_user(email)

        if user:

            stored_password = user[3]

            if bcrypt.checkpw(
                password.encode(),
                stored_password.encode()
            ):

                token = generate_token(email)

                st.session_state["logged_in"] = True
                st.session_state["token"] = token
                st.session_state["username"] = user[1]

                st.success("Login Successful!")

            else:
                st.error("Incorrect Password")

        else:
            st.error("User Not Found")

In [ ]:
## Forgot Password

Users can reset their password by receiving an OTP via Gmail SMTP, verifying it, and setting a new encrypted password.

In [ ]:
elif page == "Forgot Password":

    st.header("🔒 Forgot Password")

    email = st.text_input("Registered Email")

    if "otp" not in st.session_state:
        st.session_state.otp = ""

    if st.button("Generate OTP"):

        user = get_user(email)

        if user:

            otp = generate_otp()

            st.session_state.otp = otp
            st.session_state.reset_email = email

            if send_otp(email, otp):
                st.success("OTP Sent Successfully")
            else:
                st.error("Failed to Send OTP")

        else:
            st.error("Email Not Registered")

    entered_otp = st.text_input("Enter OTP")

    new_password = st.text_input(
        "New Password",
        type="password"
    )

    confirm_password = st.text_input(
        "Confirm Password",
        type="password"
    )

    if st.button("Reset Password"):

        if entered_otp != st.session_state.otp:
            st.error("Invalid OTP")

        elif new_password != confirm_password:
            st.error("Passwords Do Not Match")

        else:

            hashed = bcrypt.hashpw(
                new_password.encode(),
                bcrypt.gensalt()
            ).decode()

            update_password(
                st.session_state.reset_email,
                hashed
            )

            st.success("Password Updated Successfully")

## Dashboard

After successful login, users are redirected to the dashboard where they can upload CSV files.

In [ ]:
elif page == "Dashboard":

    if st.session_state.get("logged_in"):

        st.header("📊 Dashboard")

        st.success(
            f"Welcome {st.session_state['username']}"
        )

        uploaded_file = st.file_uploader(
            "Upload CSV File",
            type=["csv"]
        )
        if st.button("Logout"):

         st.session_state.clear()
        st.success("Logged Out Successfully")
    if uploaded_file:

            import pandas as pd

            df = pd.read_csv(uploaded_file)

            st.write(df)

            st.success("CSV Uploaded Successfully!")

    else:

        st.warning("Please Login First.")

## Conclusion

This milestone successfully implements:

- User Registration
- Secure Password Hashing
- Login Authentication
- JWT Token Generation
- Forgot Password with OTP
- Password Reset
- Dashboard with CSV Upload
- Neon PostgreSQL Integration